*When the jupyter file is done, try to optimize the code even more*

# Workshop 1 - data management

In [1]:
import os
new_directory = r"C:\Users\p90j\Desktop\Jakob\MedicinPred2Sandbox\Day 1\Workshop - data management-20230817\Formatted data"
os.chdir(new_directory)

current_directory = os.getcwd()
print("Current working directory:", current_directory)

Current working directory: C:\Users\p90j\Desktop\Jakob\MedicinPred2Sandbox\Day 1\Workshop - data management-20230817\Formatted data


In [2]:
import pandas as pd
import numpy as np
import random
random.seed(42)


In [3]:
data_folder = r"C:\Users\p90j\Desktop\Jakob\MedicinPred2Sandbox\Day 1\Workshop - data management-20230817\Formatted data\Data"

baseline = pd.read_csv(f"{data_folder}/baseline_data.csv")
diag = pd.read_csv(f"{data_folder}/diag_data.csv")
dict_data = pd.read_csv(f"{data_folder}/dict_data.csv")
quest = pd.read_csv(f"{data_folder}/quest_data.csv")
blood = pd.read_csv(f"{data_folder}/blood_data.csv").rename(columns={"..record.id": "id"})
treat = pd.read_csv(f"{data_folder}/treat_data.csv").rename(columns={"record_id": "id"})
visits = pd.read_csv(f"{data_folder}/visit_date.csv").rename(columns={"visit_date": "date"})
visits["visit"] = True

events = pd.read_csv(f"{data_folder}/events.csv")

### Translate diagnosis codes

In [4]:
diag = diag.merge(dict_data, on="code").drop(columns=["code"])

### Removing variable with >20% missing (for a more rigorous approach this should be evaluated based on training data only, but here for simplicity done on all available data)

In [5]:
threshold = 0.2
baseline = baseline.dropna(thresh=len(baseline) * (1 - threshold), axis=1)
blood = blood.dropna(thresh=len(blood) * (1 - threshold), axis=1)
quest = quest.dropna(thresh=len(quest) * (1 - threshold), axis=1)
treat = treat.dropna(thresh=len(treat) * (1 - threshold), axis=1)

### Training, validation, calibration, and test set

In [6]:
split_probabilities = [0.8, 0.05, 0.05, 0.1]
idx = np.random.choice(["train", "validation", "calibration", "test"], size=len(baseline), replace=True, p=split_probabilities)
splits = pd.DataFrame({"id": baseline["id"], "split": idx})

### One dataset with a row per time stamp to contain all observed features

*Make a more advanced Py file in the script folder. Much like the R file.*

In [7]:
timevar = pd.merge(blood, diag, on=["id", "sample_date"], how="outer")
timevar = timevar.drop(columns=["Unnamed: 0", "Unnamed: 0_x", "Unnamed: 0_y"])
quest.rename(columns={"date.x": "sample_date"}, inplace=True)
#print("After pd.merge(blood, diag)")
#print(timevar)

In [8]:
timevar = timevar.merge(quest, how="outer", on=["id", "sample_date"])
timevar = timevar.drop(columns=["Unnamed: 0"])
treat.rename(columns={"treat_start_date": "sample_date"}, inplace=True)
#print("After pd.merge(quest)")
#print(timevar)

In [9]:
timevar = timevar.merge(treat, how="outer", on=["id", "sample_date"])
timevar = timevar.drop(columns=["Unnamed: 0"])
#print("After pd.merge(treat)")
#print(timevar)

In [10]:
timevar = pd.merge(timevar, visits, left_on=["id", "sample_date"], right_on=["id", "date"], how="outer")
timevar = timevar.drop(columns=["Unnamed: 0"])
#print("After pd.merge(visits)")
#print(timevar)

In [11]:
timevar = pd.merge(timevar, baseline, on="id", how="outer")
#print("After pd.merge(baseline)")
#print(timevar)

In [12]:
timevar = timevar.rename(columns={"sample_date": "date"})
#print("After renaming")
#print(timevar)

In [13]:
timevar = pd.merge(timevar, splits, on="id", how="left")
#print("After pd.merge(splits)")
#print(timevar)

### Mean values and standard deviation for imputation and normalization

In [14]:
filtered_data = timevar[timevar["split"] == "train"]
#print("Filtered Data:")
#print(filtered_data)

In [15]:
data_without_diag = filtered_data.drop(columns=["diag"])

#print("Data without diag:")
#print(data_without_diag)

In [16]:
mean_columns = data_without_diag.mean()
mean_rounded_columns = data_without_diag[["systolic", "diastolic", "hemoglobin", "platelets", "creatine",
                                          "anxiety", "vegetable", "aspirin", "statins"]].mean().round()

mean_columns_pp = mean_columns.append(mean_rounded_columns)
mean_columns = mean_columns.append(mean_rounded_columns)
mean_columns.rename(index=lambda col: f"mean_{col}", inplace=True)


#print("Mean Columns:")
#print(mean_columns)

C:\Users\p90j\AppData\Local\Temp\ipykernel_4804\2602731296.py:1: FutureWarning: The default value of numeric_only in DataFrame.mean is deprecated. In a future version, it will default to False. In addition, specifying 'numeric_only=None' is deprecated. Select only valid columns or specify the value of numeric_only to silence this warning.
  mean_columns = data_without_diag.mean()
C:\Users\p90j\AppData\Local\Temp\ipykernel_4804\2602731296.py:5: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mean_columns_pp = mean_columns.append(mean_rounded_columns)
C:\Users\p90j\AppData\Local\Temp\ipykernel_4804\2602731296.py:6: FutureWarning: The series.append method is deprecated and will be removed from pandas in a future version. Use pandas.concat instead.
  mean_columns = mean_columns.append(mean_rounded_columns)


In [17]:
q1_columns = data_without_diag.quantile(0.01)

q1_columns.rename(index=lambda col: f"q1_{col}", inplace=True)

#print("Q1 Columns:")
#print(q1_columns)

C:\Users\p90j\AppData\Local\Temp\ipykernel_4804\2822825231.py:1: FutureWarning: The default value of numeric_only in DataFrame.quantile is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  q1_columns = data_without_diag.quantile(0.01)


In [18]:
q99_columns = data_without_diag.quantile(0.99)

q99_columns.rename(index=lambda col: f"q99_{col}", inplace=True)

#print("Q99 Columns:")
#print(q99_columns)

C:\Users\p90j\AppData\Local\Temp\ipykernel_4804\2202733825.py:1: FutureWarning: The default value of numeric_only in DataFrame.quantile is deprecated. In a future version, it will default to False. Select only valid columns or specify the value of numeric_only to silence this warning.
  q99_columns = data_without_diag.quantile(0.99)


In [19]:
summaries = pd.concat([mean_columns, q1_columns, q99_columns])

#print("Final Summaries:")
#print(summaries)

### Feature engineering

In [20]:
timevar = timevar.loc[:,~timevar.columns.duplicated()].copy()

timevar["date"] = pd.to_datetime(timevar["date"])
timevar["d.birth"] = pd.to_datetime(timevar["d.birth"])

In [21]:
final_data = timevar.sort_values(by=["id", "date"])

#print("Data after arranging:")
#print(final_data)

In [22]:
def calculate_age(row):
    birth_date = pd.Timestamp(row["d.birth"])
    return (row["date"] - birth_date).days / 365.25 #leap year 

final_data["visit"] = final_data["visit"].fillna(False)
final_data["age"] = final_data.apply(calculate_age, axis=1)
final_data["male"] = np.where(final_data['sex'] == "male", 1, 0)
final_data["smoker_current"] = np.where(final_data["smoking"] == "current", 1, 0)
final_data["smoker_former"] = np.where(final_data["smoking"] == "former", 1, 0)

#print("Data after mutation:")
#print(final_data)


In [23]:
diagnoses = ["diabetes", "hyperlipidemia", "hypertension"]
for diagnosis in diagnoses:
    final_data[diagnosis] = np.where(final_data["diag"] == diagnosis, 1, 0)
    final_data[diagnosis] = final_data.groupby("id")[diagnosis].cumsum()

In [24]:
final_data = final_data.drop(columns=["diag", "d.birth", "sex", "smoking"])
columns_to_fill = ["systolic", "diastolic", "ldl", "hdl", "statins"]
final_data[columns_to_fill] = final_data[columns_to_fill].fillna(method='ffill')

#print("Data after column selection and filling missing values:")
#print(final_data)

In [25]:
final_data = final_data.copy()
final_data.fillna(mean_columns_pp, inplace=True) 



#print("Data after replacing missing values with means:")
#print(final_data)

In [26]:
start_column = "systolic"
end_column = "statins"

columns_to_transform = final_data.columns[final_data.columns.get_loc("systolic"):final_data.columns.get_loc("statins")+1]

for col in columns_to_transform:
    final_data[col] = final_data[col].apply(lambda x: max(x, summaries[f"q1_{col}"]))
    final_data[col] = final_data[col].apply(lambda x: min(x, summaries[f"q99_{col}"]))
    final_data[col] = (final_data[col] - final_data[col].min()) / (final_data[col].max() - final_data[col].min())

#print("Data after capping and normalizing values:")
#print(final_data)

In [27]:
final_data = final_data[final_data["visit"]]
final_data.drop(columns=["visit"], inplace=True)
final_data = final_data[["id", "split"] + [col for col in final_data.columns if col not in ["id", "split"]]]

#print("Data after filtering and selecting columns:")
#print(final_data)

In [28]:
final_data["date"] = pd.to_datetime(final_data["date"])
events["date"] = pd.to_datetime(events["date"])

deaths = events[events["type"] == "death"].rename(columns={"date": "d.death"})[["d.death", "id"]]

final_data = pd.merge(final_data, deaths, on="id", how="left")

final_data["died_1y"] = ((final_data["d.death"] - final_data["date"]) / pd.Timedelta(days=365)) <= 1
final_data["died_1y"] = final_data["died_1y"].astype(int)

#print("Data after joining with events and calculating 1-year mortality:")
#print(final_data)

In [29]:
final_data.drop(columns=["d.death"], inplace=True)

print("Final Data:")
print(final_data)
#print(final_data.describe())
#print(final_data.shape)

Final Data:
        id        split       date  systolic  diastolic  erythrocytes  \
0        2  calibration 2005-06-10  0.458422   0.349650      0.805812   
1        2  calibration 2006-09-12  0.245203   0.271950      0.805812   
2        2  calibration 2007-10-15  0.692964   0.271950      0.805812   
3        2  calibration 2008-09-22  0.692964   0.757576      0.717546   
4        2  calibration 2009-09-21  0.479744   0.466200      0.805812   
...    ...          ...        ...       ...        ...           ...   
3904  1592  calibration 2007-06-09  0.394456   0.349650      0.774381   
3905  1593        train 2005-11-07  0.479744   0.660451      0.805812   
3906  1594        train 2005-10-03  0.586354   0.660451      0.805812   
3907  1595        train 2005-06-02  0.522388   0.505051      0.840097   
3908  1596        train 2005-12-20  0.479744   0.310800      0.805812   

      hemoglobin       wbc  platelets   glucose  ...  ezetimibe  statins  \
0       0.808252  0.538345   0.5423

Now where the data is ready for the model, it can be saved as a csv or any other preferred format. 

In [30]:
final_data.to_csv("final_data.csv")